In [68]:
import ROOT

import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as interp
import scipy.optimize as opt
import matplotlib as mpl
import scipy.special as sf
import scipy.integrate as integ
import scipy.linalg as la
mpl.rc('xtick', direction='in', top=True)
mpl.rc('ytick', direction='in', right=True)
mpl.rc('xtick.minor', visible=True)
mpl.rc('ytick.minor', visible=True)
import pandas as pd
import os
import re
import uuid
import json
import copy



In [69]:
base_path1="Band_class_training data/TE_011/Results"
sidebandspath1 = os.path.join(base_path1, "Te011_sidebands.json")
with open(sidebandspath1, "r") as f:
    TE011_Sidebands_Index = json.load(f)
maxbandspath1 = os.path.join(base_path1, "Te011_MaxBands.json")
with open(maxbandspath1, "r") as f:
    TE011_Maxbands_Index = json.load(f)


In [70]:
base_path2="Band_class_training data/TE_012/Results"
sidebandspath2 = os.path.join(base_path2, "Te012_sidebands.json")
with open(sidebandspath2, "r") as f:
    TE012_Sidebands_Index = json.load(f)
maxbandspath2 = os.path.join(base_path2, "Te012_MaxBands.json")
with open(maxbandspath2, "r") as f:
    TE012_Maxbands_Index = json.load(f)


In [71]:
def make_inputs(index,numpeaks=10,numfeatures=2):
    first_angle = next(iter(index))
    first_position = next(iter(index[first_angle]))

    Npsd=len(index)*len(index[first_angle])*(len(index[first_angle][first_position]))
    
    empt=np.zeros((Npsd,numpeaks,numfeatures))
    jsn_name = "LocustEventProperties.json"
    band_name='side_bands_powercenter'
    i=0
    for angle in index:
        for position in index[angle]:
            for hist in index[angle][position]:
                 if hist != jsn_name:
                     peaks=np.array(index[angle][position][hist][band_name])
                     empt[i,:,:]=peaks
                     i=i+1
    full=empt
    return(full)

In [125]:
def make_outputs(max_array,band_array):
    empt = np.zeros(max_array.shape[:2])
    for i in range(len(max_array)):
        ar1=max_array[i]
        ar2=band_array[i]
        match_matrix = (ar1[:, None, :] == ar2[None, :, :2]).all(axis=2)
        ar1_i, ar2_i = np.where(match_matrix)
        out = np.full(ar1.shape[0], np.inf)
        out[ar1_i] = ar2[ar2_i, 2]
        empt[i]=out
    return(empt)

    


In [73]:
max1=make_inputs(TE011_Maxbands_Index)
max2=make_inputs(TE012_Maxbands_Index)

In [74]:
band1=make_inputs(TE011_Sidebands_Index,numpeaks=5,numfeatures=3)
band2=make_inputs(TE012_Sidebands_Index,numpeaks=4,numfeatures=3)

In [126]:
out1=make_outputs(max1,band1)
out2=make_outputs(max2,band2)